In [1]:
%%time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType
import pandas as pd
import glow

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

hostname = !hostname
spark = SparkSession \
    .builder \
    .master(f'spark://{hostname[0]}:7077') \
    .appName('glow_pyspark') \
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-spark_2.12:3.1.0,io.projectglow:glow-spark3_2.12:2.0.0') \
    .config('spark.jars.excludes', 'org.apache.hadoop:hadoop-client,io.netty:netty-all,io.netty:netty-handler,io.netty:netty-transport-native-epoll') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .config('spark.sql.debug.maxToStringFields', '0') \
    .config('spark.hadoop.io.compression.codecs', 'io.projectglow.sql.util.BGZFCodec') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider') \
    .config('spark.executor.memory', '15g') \
    .config('spark.driver.memory', '10g') \
    .config('spark.driver.cores', '2') \
    .getOrCreate()
spark = glow.register(spark)

:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
io.projectglow#glow-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2f5a9ef2-e0be-4c44-9154-599ad9a536e8;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found io.projectglow#glow-spark3_2.12;2.0.0 in central
	found org.seqdoop#hadoop-bam;7.10.0 in central
	found com.github.jsr203hadoop#jsr203hadoop;1.0.3 in central
	found org.slf4j#slf4j-api;2.0.12 in central
	found org.jdbi#jdbi;2.78 in central
	found com.github.broadinstitute#pica

CPU times: user 2.44 s, sys: 329 ms, total: 2.77 s
Wall time: 17.9 s


In [41]:
%%time
consequences_to_keep = [
    'transcript ablation',
    'stop gained',
    'frameshift',
    'stop lost',
    'start lost',
    'transcript amplification',
    'inframe insertion',
    'inframe deletion',
    'missense',
    'protein altering'
]

germline_var = spark \
    .read \
    .csv("/sbgenomics/project-files/germline_somatic_gene_filtering_results/", sep="\t", nullValue='-', header=True, inferSchema=True)

# Convert the stringified list into an actual array
germline_var = germline_var.withColumn(
    "consequence_array",
    F.from_json("consequence", ArrayType(StringType()))
)

# Filter rows where intersection length > 0
germline_var_filtered = germline_var.where(
                                (F.size(
                                    F.array_intersect(
                                        F.col('consequence_array'),
                                        F.array(*[F.lit(x) for x in consequences_to_keep])
                                    )
                                ) > 0) & ~F.col('hgvsp').isNull() & 
                                F.col("ensembl_gene_id").startswith("ENSG")
                            ) \
                        .withColumn('hgvsp_arr', F.split('hgvsp', ':')) \
                        .withColumn('name', F.col('hgvsp_arr').getItem(1))

germline_var_filtered.limit(2).toPandas()

CPU times: user 33.6 ms, sys: 0 ns, total: 33.6 ms
Wall time: 11.9 s


,study_id,participant_id,chromosome,start,reference,alternate,consequence,vep_impact,symbol,ensembl_gene_id,refseq_mrna_id,hgvsc,hgvsp,topmed_bravo_af,gnomad_genomes_2_1_1_af,gnomad_exomes_2_1_1_af,gnomad_genomes_3_af,max_gnomad_topmed,DamagePredCount,PredCountRatio_D2T,TWINSUK_AF,ALSPAC_AF,UK10K_AF,HGMDID,variant_class,phen,ad_ref,ad_alt,dp,variant_allele_fraction,calls,adjusted_calls,filters,is_lo_conf_denovo,is_hi_conf_denovo,is_proband,affected_status,gender,sample_id,mother_id,father_id,family_id,diagnoses_combined,phenotypes_combined,hpos_combined,study_code,consequence_array,hgvsp_arr,name
0,SD_BHJXBDQK,PT_SZJ7WZZW,1,999085,CACCCGGGCCCTGCGGCCCCGCCCTGGGGGCGGCGGGCAGCGCCCGGGTCAG,C,['inframe deletion'],MODERATE,HES4,ENSG00000188290,None,ENST00000428771.6:c.667_717del,ENSP00000393198.2:p.Leu223_Gly239del,NaN,NaN,NaN,NaN,0.000000,None,NaN,NaN,NaN,NaN,None,None,None,13,10,23.0,0.434783,"[0, 1]","[0, 1]",['PASS'],None,None,True,False,male,BS_FHDR7FJE,None,None,FM_S683H4KD,"['Medulloblastoma, Group 4']","['Medulloblastoma', 'Medulloblastoma, NOS or NEC']","['HP:0002885', 'HP:0002885']",CBTN,[inframe deletion],"[ENSP00000393198.2, p.Leu223_Gly239del]",p.Leu223_Gly239del
1,SD_BHJXBDQK,PT_C3NQTXD7,1,999364,C,T,['missense'],MODERATE,HES4,ENSG00000188290,['NM_021170.4'],ENST00000304952.11:c.361G>A,ENSP00000304595.7:p.Val121Met,0.000008,NaN,0.000078,NaN,0.000078,14_21,0.666667,NaN,NaN,NaN,None,None,None,20,19,39.0,0.487179,"[0, 1]","[0, 1]",['PASS'],None,None,True,False,male,BS_Z2JFHMWQ,PT_T90R0Y9M,PT_DXW38XQH,FM_HC41F2TQ,"['Glioneuronal and neuronal tumors, Ganglioglioma', 'Low-Grade Glioma, NOS or NEC']","['Glioneuronal and neuronal tumors, Ganglioglioma', 'Glioma', 'Low-Grade Glioma, NOS or NEC', 'Ganglioglioma']","['HP:0033664', 'HP:0009733', 'HP:0009733', 'HP:0033664']",CBTN,[missense],"[ENSP00000304595.7, p.Val121Met]",p.Val121Met


In [3]:
num_ptids = germline_var_filtered.select("participant_id").distinct().count()
print(f"👤 Unique participant IDs: {num_ptids}")
num_genes = germline_var_filtered.select("symbol").distinct().count()
print(f"🧬 Unique genes: {num_genes}")

👤 Unique participant IDs: 1719


🧬 Unique genes: 13066


In [42]:
%%time
somatic_var_filtered = spark \
    .read \
    .format('parquet') \
    .load('/sbgenomics/project-files/part-00000-6e6b1b42-5db9-4910-a0ac-e8bf31036fb0-c000.snappy.parquet') \
    .where(~F.col('HGVSp').isNull())
somatic_var_filtered.limit(2).toPandas()

CPU times: user 21.4 ms, sys: 3.73 ms, total: 25.1 ms
Wall time: 1.51 s


,participant_id,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,dbSNP_RS,dbSNP_Val_Status,Tumor_Sample_Barcode,Matched_Norm_Sample_Barcode,Match_Norm_Seq_Allele1,Match_Norm_Seq_Allele2,Tumor_Validation_Allele1,Tumor_Validation_Allele2,Match_Norm_Validation_Allele1,Match_Norm_Validation_Allele2,Verification_Status,Validation_Status,Mutation_Status,Sequencing_Phase,Sequence_Source,Validation_Method,Score,BAM_File,Sequencer,Tumor_Sample_UUID,Matched_Norm_Sample_UUID,HGVSc,HGVSp,HGVSp_Short,Transcript_ID,Exon_Number,t_depth,t_ref_count,t_alt_count,n_depth,n_ref_count,n_alt_count,all_effects,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,ALLELE_NUM,DISTANCE,STRAND_VEP,SYMBOL,SYMBOL_SOURCE,BIOTYPE,CANONICAL,CCDS,ENSP,SWISSPROT,TREMBL,UNIPARC,RefSeq,SIFT,PolyPhen,EXON,INTRON,DOMAINS,AF,AFR_AF,AMR_AF,ASN_AF,EAS_AF,EUR_AF,SAS_AF,AA_AF,EA_AF,CLIN_SIG,SOMATIC,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,IMPACT,PICK,VARIANT_CLASS,TSL,HGVS_OFFSET,PHENO,MINIMISED,GENE_PHENO,FILTER,flanking_bps,vcf_id,vcf_qual,gnomAD_AF,gnomAD_AFR_AF,gnomAD_AMR_AF,gnomAD_ASJ_AF,gnomAD_EAS_AF,gnomAD_FIN_AF,gnomAD_NFE_AF,gnomAD_OTH_AF,gnomAD_SAS_AF,HGVSg,vcf_pos,gnomad_3_1_1_AC,gnomad_3_1_1_AN,gnomad_3_1_1_AF,gnomad_3_1_1_nhomalt,gnomad_3_1_1_AC_popmax,gnomad_3_1_1_AN_popmax,gnomad_3_1_1_AF_popmax,gnomad_3_1_1_nhomalt_popmax,gnomad_3_1_1_AC_controls_and_biobanks,gnomad_3_1_1_AN_controls_and_biobanks,gnomad_3_1_1_AF_controls_and_biobanks,gnomad_3_1_1_AF_non_cancer,gnomad_3_1_1_primate_ai_score,gnomad_3_1_1_splice_ai_consequence,MQ,MQ0,CAL,HotSpotAllele,official_gene_symbol
0,PT_SABX4PZ2,JUN,0,None,GRCh38,chr1,58782452,58782454,+,In_Frame_Del,DEL,GCT,GCT,-,rs748534142,None,BS_41Q1D6PV,BS_16C081MC,GCT,GCT,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,c.617_619del,p.Gln206del,p.Q206del,ENST00000371222,1/1,131,117,14,160,160,0,"JUN,inframe_deletion,p.Gln206del,ENST00000371222,NM_002228.4;JUN,inframe_deletion,p.Gln206del,NM_002228.4,;LINC01135,upstream_gene_variant,,ENST00000419531,;LINC01135,upstream_gene_variant,,ENST00000649834,;LINC01135,upstream_gene_variant,,ENST00000653297,;LINC01135,upstream_gene_variant,,ENST00000663144,;LINC01135,upstream_gene_variant,,ENST00000669294,;LINC01135,upstream_gene_variant,,ENST00000685887,;LINC01135,upstream_gene_variant,,ENST00000687989,;LINC01135,upstream_gene_variant,,ENST00000693275,;LINC01135,upstream_gene_variant,,NR_034014.1,;LINC01135,upstream_gene_variant,,NR_034015.1,;LINC01135,upstream_gene_variant,,NR_108106.1,;JUN,inframe_deletion,p.Gln206del,ENST00000678696,;,regulatory_region_variant,,ENSR00000007152,;",-,ENSG00000177606,ENST00000371222,Transcript,inframe_deletion,1594-1596/3257,617-619/996,206-207/331,QP/P,cAGCcg/ccg,rs748534142,1,NaN,-1,JUN,HGNC,protein_coding,YES,CCDS610.1,ENSP00000360266,P05412.255,None,UPI000000D908,NM_002228.4,None,None,1/1,None,"Pfam:PF03957,PANTHER:PTHR11462,PANTHER:PTHR11462:SF8,Low_complexity_(Seg):seg",None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,MODERATE,1,deletion,NaN,NaN,None,None,1,PASS,CGGCTG,None,None,None,None,None,None,None,None,None,None,None,chr1:g.58782465_58782467del,58782451,1.0,152148.0,0.000007,0.0,1.0,67986.0,0.000015,0.0,1.0,32874.0,0.00003,0.000007,NaN,no_consequence,60,0,"Strelka2,VarDict,Lancet",0,JUN
1,PT_SABX4PZ2,ACVR1,0,None,GRCh38,chr2,157774114,157774114,+,Missense_Mutation,SNP,C,C,T,rs121912678,None,BS_41Q1D6PV,BS_16C081MC,C,C,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,c.617G>A,p.Arg206His,p.R206H,ENST00000434821,6/11,350,246,104,387,387,0,"ACVR1,missense_variant,p.Arg206His,XM_011512108.3,;ACVR1,missense_variant,p.Arg206His,ENST00000682300,;ACVR1,missense_variant,p.Arg206His,ENST00000683487,;ACVR1,missense_variant,p.Arg206His,ENST00000434821,NM

In [5]:
num_ptids = somatic_var_filtered.select("participant_id").distinct().count()
print(f"👤 Unique participant IDs: {num_ptids}")
num_genes = somatic_var_filtered.select("symbol").distinct().count()
print(f"🧬 Unique genes: {num_genes}")

👤 Unique participant IDs: 2554
🧬 Unique genes: 18513


In [6]:
matched_gene_participants = germline_var_filtered.select("participant_id", "symbol", "name").distinct() \
    .groupBy("participant_id", "symbol") \
    .agg(
        F.concat_ws(";", F.collect_list("name")).alias("germline_hgvsps")
    ) \
    .join(
        somatic_var_filtered.select("participant_id", "official_gene_symbol", "hgvsp").distinct()  \
            .groupBy("participant_id", "official_gene_symbol") \
            .agg(
            F.concat_ws(";", F.collect_list("hgvsp")).alias("somatic_hgvsps")
        ),
        on="participant_id",
        how="inner"
    ) \
    .where(F.col("symbol") == F.col("official_gene_symbol")) \
    .distinct()
num_gp_pair= matched_gene_participants.count()
print(f"Number of the gene-participant pair: {num_gp_pair}")
matched_gene_participants.limit(2).toPandas()

Number of the gene-participant pair: 1060


,participant_id,symbol,germline_hgvsps,official_gene_symbol,somatic_hgvsps
0,PT_0SPKM4S8,ALPI,p.Ala350Val,ALPI,p.Leu272Ile
1,PT_0SPKM4S8,BRCA2,p.Val2466Ala,BRCA2,p.Asp3108Tyr;p.Asn978Thr


In [8]:
%%time
# average numbers for both germline and somatic variants in all the gene-participant pairs 
# average_germline_variant_count = matched_gene_participants.agg(F.avg("germline_variant_count").alias("average_germline_variant_count"))
# average_somatic_variant_count = matched_gene_participants.agg(F.avg("somatic_variant_count").alias("somatic_variant_count"))

# Show the result
# average_germline_variant_count.show()
# average_somatic_variant_count.show()

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 7.39 µs


In [9]:
%%time
import ipywidgets as widgets
from IPython.display import display

# Get unique gene symbols from df1 (efficient alternative)
gene_list = [row['symbol'] for row in matched_gene_participants.select('symbol').distinct().collect()]
gene_list.sort()  # Optional

# Create a combobox (searchable dropdown)
gene_selector = widgets.Combobox(
    placeholder='Type or select a gene',
    options=gene_list,
    description='Gene:',
    ensure_option=True,  # Only allow valid options
    style={'description_width': 'initial'},
)

# Variable to store selected gene
gene_selected = gene_selector.value

# Update function
def on_gene_change(change):
    global gene_selected
    if change['type'] == 'change' and change['name'] == 'value':
        gene_selected = change['new']
        print(f"Selected gene: {gene_selected}")

# Attach event handler
gene_selector.observe(on_gene_change)

# Display the widget
display(gene_selector)

Combobox(value='', description='Gene:', ensure_option=True, options=('ABCA10', 'ABCA13', 'ABCA4', 'ABCA7', 'AB…

CPU times: user 74.6 ms, sys: 1.05 ms, total: 75.6 ms
Wall time: 6.58 s


In [10]:
%%time
print(f"you selected gene", gene_selected)

you selected gene BRCA2
CPU times: user 241 µs, sys: 0 ns, total: 241 µs
Wall time: 176 µs


In [11]:
%%time
# for this same gene_selected, get a PT_ID list that have both germline and somatic variants
# and let the user pick one ptid_selected from this list
pt_list = [row['participant_id'] for row in matched_gene_participants \
           .where(F.col('symbol') == gene_selected) \
           .select('participant_id').distinct().collect()]
pt_list.sort()  # Optional

# Create a combobox (searchable dropdown)
pt_selector = widgets.Combobox(
    placeholder='Type or select a particiant ID',
    options=pt_list,
    description='Particiant ID:',
    ensure_option=True,  # Only allow valid options
    style={'description_width': 'initial'},
)

# Variable to store selected gene
pt_selected = pt_selector.value

# Update function
def on_pt_change(change):
    global pt_selected
    if change['type'] == 'change' and change['name'] == 'value':
        pt_selected = change['new']
        print(f"Selected gene: {pt_selected}")

# Attach event handler
pt_selector.observe(on_pt_change)

# Display the widget
display(pt_selector)

Combobox(value='', description='Particiant ID:', ensure_option=True, options=('PT_0SPKM4S8', 'PT_3CHB9PK5', 'P…

CPU times: user 12.5 ms, sys: 3.65 ms, total: 16.1 ms
Wall time: 5.58 s


In [12]:
%%time
print(f"you selected participant", pt_selected)

you selected participant PT_S0Q27J13
CPU times: user 64 µs, sys: 7 µs, total: 71 µs
Wall time: 52.2 µs


In [13]:
%%time
matched_gene_participants.limit(5).toPandas()

CPU times: user 5.76 ms, sys: 7.89 ms, total: 13.6 ms
Wall time: 6.31 s


,participant_id,symbol,germline_hgvsps,official_gene_symbol,somatic_hgvsps
0,PT_0SPKM4S8,ALPI,p.Ala350Val,ALPI,p.Leu272Ile
1,PT_0SPKM4S8,BRCA2,p.Val2466Ala,BRCA2,p.Asp3108Tyr;p.Asn978Thr
2,PT_0SPKM4S8,C6orf89,p.Ile42Thr,C6orf89,p.Ala66Thr
3,PT_0SPKM4S8,CLCN1,p.Gly118Trp,CLCN1,p.Lys783Arg
4,PT_0SPKM4S8,COL4A5,p.Ile444Ser,COL4A5,p.Ser164Pro;p.Gly423Val


In [14]:
%%time
matched_gene_participants.groupBy('official_gene_symbol').count().orderBy(F.desc('count')).limit(10).toPandas()

CPU times: user 12.4 ms, sys: 1.4 ms, total: 13.8 ms
Wall time: 5.9 s


,official_gene_symbol,count
0,AHNAK2,147
1,KMT2C,39
2,PCLO,23
3,MUC3A,14
4,TTN,13
5,PRSS1,12
6,KIF1A,11
7,DMD,9
8,HLA-A,9
9,NOTCH3,7


In [15]:
%%time
somatic_var_short = somatic_var_filtered.where((F.col("participant_id") == pt_selected) & \
                           (F.col("official_gene_symbol") == gene_selected)) \
                    .withColumn("codon_pos", F.regexp_extract("hgvsp", r"(\d+)", 1)) \
                    .withColumnRenamed("hgvsp", "name") \
                    .withColumn("source", F.lit("somatic")) \
                    .select("name", "codon_pos", "source") \
                    .distinct()
germline_var_short = germline_var_filtered.where((F.col("participant_id") == pt_selected) & \
                           (F.col("symbol") == gene_selected)) \
                    .withColumn("codon_pos", F.regexp_extract("name", r"(\d+)", 1)) \
                    .withColumn("source", F.lit("germline")) \
                    .select("name", "codon_pos", "source") \
                    .distinct()
merged_germ_somatic_vars = germline_var_short.unionByName(somatic_var_short)
output_file = gene_selected + "_" + pt_selected + "_merged_germ_somatic_vars.tsv"
merged_germ_somatic_vars.toPandas().to_csv(
    output_file, sep='\t', header=True, index=False)

CPU times: user 23.5 ms, sys: 3.97 ms, total: 27.5 ms
Wall time: 5.34 s


In [16]:
%%time
import subprocess
from IPython.display import SVG, display
import os

# Run the fetch_protein_domain.sh 
command1 = ["bash", "scripts/fetch_protein_domain.sh", gene_selected]
result1 = subprocess.run(command1, capture_output=True, text=True)
# Print the command and result
print(f"Running command: {command1}")

# If the command was successful, result.returncode will be 0
if result1.returncode != 0:
    # Print any error message (stderr)
    print(f"Error occurred: {result1.stderr}")
else:
    # Otherwise, print the normal output (stdout)
    print(f"Output: {result1.stdout}")

Running command: ['bash', 'fetch_protein_domain.sh', 'BRCA2']
Output: Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,543 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,155 kB]
Fetched 4,955 kB in 7s (700 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
jq is already the newest version (1.6-2.1ubuntu3).
0 upgraded, 0 newly installed, 0 to remove and 101 not upgraded.
Fetching Pfam domains for UniProt accession: P51587

CPU times: user 5.02 ms, sys: 123 µs, total: 5.14 ms
Wall time: 10.8 s


In [40]:
%%time
# Run the lollipops command
# if this is your first time running generate_lollipop_plot.R, it might take about 40 mins as some packages need to be installed
command2 = ["Rscript","scripts/generate_lollipop_plot_trackViewer.r", gene_selected, pt_selected]
result2 = subprocess.run(command2, capture_output=True, text=True)
# Print the command and result
print(f"Running command: {command2}")

# If the command was successful, result.returncode will be 0
if result2.returncode != 0:
    # Print any error message (stderr)
    print(f"Error occurred: {result2.stderr}")
else:
    # Otherwise, print the normal output (stdout)
    print(f"Output: {result2.stdout}")

Running command: ['Rscript', 'scripts/generate_lollipop_plot_trackViewer.r', 'NF1', 'PT_2BJF83GQ']
Output: x86_64-conda-linux-gnu-cc -I"/opt/conda/lib/R/include" -DNDEBUG  -I'/opt/conda/lib/R/library/S4Vectors/include' -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /opt/conda/include -I/opt/conda/include -Wl,-rpath-link,/opt/conda/lib    -fpic  -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /opt/conda/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1695968301569/work=/usr/local/src/conda/r-base-4.3.1 -fdebug-prefix-map=/opt/conda=/usr/local/src/conda-prefix  -c R_init_DelayedArray.c -o R_init_DelayedArray.o
x86_64-conda-linux-gnu-cc -I"/opt/conda/lib/R/include" -DNDEBUG  -I'/opt/conda/lib/R/library/S4Vectors/include' -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /opt/conda/include -I/opt/conda/include -Wl,-rpath-link,/opt/conda/lib    -fpic  -march=nocona -mtune=haswell -ftree-vector

In [18]:
%%time
# displaying the lollipop plot
# germline varint(s) shows on the top and somatic variant(s) shows below.
display(SVG(filename=f"{gene_selected}_{pt_selected}_lollipop.svg"))

FileNotFoundError: [Errno 2] No such file or directory: 'BRCA2_PT_S0Q27J13_lollipop.svg'